# Attempt at Week2 Challenge
## Transforming Deep Research into an Agentic Flow
### We will attempt to create three top level agents: Preparer, Reporter, and Verifier
Perparer:

    * Generates 3 Questions for the topic
    * Generates 3 Search Strings for the topic


In [19]:
from agents import Agent, Runner, trace, AsyncOpenAI, OpenAIChatCompletionsModel, function_tool, WebSearchTool
from pydantic import BaseModel, Field
from typing import List

### Define the topic to research

In [2]:
topic = "Kobe Bryant"

## Use Ollama for WebSearch to minimize costs (gpt charges more for using the WebSearchTool)


In [7]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
o_client = AsyncOpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")
o_model = OpenAIChatCompletionsModel(model="llama3.2", openai_client=o_client)

### Create WebSearchTool for use with an ollama agent

In [8]:
@function_tool
def custom_websearch_tool():
    """Custom WebSearchTool"""
    return WebSearchTool(search_context_size="low")


### Create the ollama WebSearch Agent
Also test the agent before moving forward

In [10]:
search_agent_instruction = "You are researcher.  You are provided a topic and are required to use your websearch tool to gather information on the topic"
search_agent = Agent(name="search_agent", instructions=search_agent_instruction, tools=[custom_websearch_tool], model=o_model)

# test it
with trace("WebSearch-Test"):
    result = await Runner.run(search_agent, topic)
    print(result.final_output)

**Summary of Kobe Bryant**

Kobe Bryant (August 23, 1978 - January 26, 2020) was an American professional basketball player who played his entire 20-year career with the Los Angeles Lakers in the National Basketball Association (NBA). He is widely regarded as one of the greatest basketball players of all time.

**Early Life and Career**

Bryant was born in Philadelphia, Pennsylvania, to Joe Bryant, a former NBA player, and Pam Shults. His family moved to Italy when he was six months old, where his father played professionally. Bryant spoke fluent Italian by the age of three. He learned basketball skills from his father and began playing professionally at 17.

**NBA Career**

The Charlotte Hornets selected Bryant with the 13th overall pick in the 1996 NBA draft, but he was immediately traded to Los Angeles. Over his career, Bryant won five NBA championships (2000-2002, 2009) and two NBA Finals MVP awards (2002, 2009). He was also an 18-time All-Star and won two regular-season scoring ti

## Now let's define and test a Questioner Agent

In [ ]:
# define output types
class Questions(BaseModel):
    questions: List[str]
    """The questions to be asked and tested for"""

questioner_agent_instruction = f"You are given a topic and are tasked with generating a very diffiult question pertaining to that topic."
questioner_agent = Agent(name="questioner", instructions=questioner_agent_instruction, model="gpt-4.1-mini", output_type=Questions)

with trace("Questioner-Test"):
    result = await Runner.run(questioner_agent, topic)
    print(result.final_output)

questions=["Analyze the impact of Kobe Bryant's 'Mamba Mentality' on the evolution of professional basketball training methodologies and athlete mindset development.", "Discuss the influence of Kobe Bryant's international career, particularly his performance and leadership in the 2008 and 2012 Olympics, on the global perception of basketball and its growth outside the United States.", "Examine the complexity of Kobe Bryant's legacy in light of both his extraordinary achievements on the court and the controversies off the court, considering the broader implications for athlete role models in contemporary society."]


### Create the search_planner_agent

In [ ]:
# define output types
class Search(BaseModel):
    search_string: str = Field("The suggested string to search for")
    reason: str = Field("Why you think this string is valuable to search for")

class SearchPlan(BaseModel):
    searches: List[Search]
    """A list of strings used to search the web for learning purposes"""
    
search_planner_instructions = f"You are a helpful research assistant. Given a topic, come up with a web search string;  Do not actually search the web, just tell me what you would search for on the web"
search_planner_agent = Agent(name="search_planner", instructions=search_planner_instructions, model="gpt-4.1-mini", output_type=SearchPlan)

# test it
with trace("Search-String-Test"):
    result = await Runner.run(search_planner_agent, topic)
    print(result.final_output)

searches=[Search(search_string='Kobe Bryant biography', reason="To get an overview of Kobe Bryant's life and career."), Search(search_string='Kobe Bryant career achievements and statistics', reason='To find detailed information on his basketball records and milestones.'), Search(search_string='Kobe Bryant tragic helicopter accident details', reason='To understand the circumstances and impact of his death.'), Search(search_string='Kobe Bryant legacy and influence on basketball', reason='To explore how he influenced the sport and other players.'), Search(search_string='Kobe Bryant famous quotes and interviews', reason='To learn about his personality, mindset, and philosophy from his own words.')]


## Now we create the Preparer Agent (using search_planner and questioner agents as tools)

In [22]:
NUM_QUESTIONS = 3
NUM_SEARCH_STRS = 5

preparer_agent_instructions = f"""
You are an assistant preparing your worker to perform some deep research on a topic.
You are tasked with coming up with {NUM_QUESTIONS} very difficult questions related to this topic. 
In addition, you are also tasked with preparing {NUM_SEARCH_STRS} search strings for your worker to use during their research
"""
preparer_agent_tools=[
    questioner_agent.as_tool(tool_name="question_tool", tool_description="this tool is used to generated questions related to a given topic"),
    search_planner_agent.as_tool(tool_name="search_planner_tool", tool_description="this tool is used to generated searching strings for a given topic"),
]
preparer_agent = Agent(name="preparer", instructions=preparer_agent_instructions, model="gpt-4.1-mini", tools=preparer_agent_tools)

with trace("preparer-agent-test"):
    result = await Runner.run(preparer_agent, topic)
    print(result.final_output)

Here are 3 very difficult questions related to Kobe Bryant for deep research:

1. Analyze the evolution of Kobe Bryant's playing style from his early years in the NBA to his final season, highlighting the key adjustments he made to maintain elite performance.  
2. Discuss the impact of Kobe Bryant's 'Mamba Mentality' on modern basketball culture and athlete mindset, providing examples of how it has influenced players beyond the sport of basketball.  
3. Evaluate Kobe Bryant's contributions to international basketball and how his global presence affected the NBA's popularity worldwide, particularly focusing on specific initiatives or events he participated in.  

Additionally, here are 5 focused search strings for research:

1. Kobe Bryant biography  
2. Kobe Bryant NBA achievements  
3. Kobe Bryant tragic helicopter accident  
4. Kobe Bryant philanthropy and legacy  
5. Kobe Bryant documentaries and interviews  

These questions and search strings will guide a thorough and deep explora